# Week 5 matched-seed ResNet replication

**Question.** Is the validation advantage of the multi-scale ResNet over the standard Week 5 ResNet stable across random initialization and minibatch order? The earlier single-checkpoint AP difference was only +0.0024. We train **both architectures afresh** at seeds 42, 43, and 44 using the same records, lead mapping, first 10 seconds, normalization, Adam settings, unweighted cross entropy, batch size, maximum 8 epochs, and minimum-validation-loss checkpoint rule. An independent seeded DataLoader generator gives the same record ordering to both architectures at each seed. Runs are sequential; every checkpoint and epoch history is preserved. The historical Week 4/5/8 artifacts are read only.

The target is exact matching to the 19 selected `ICD-10 Code` strings in `Data/DiseaseCode.csv` within the metadata `ICD-10 code` field. Negative means none of these selected codes; neither class is defined by ECG diagnostic statements. The sorted-ID seed-42 patient split is fixed at 70/15/15. Held-out patient IDs are identified but their metadata outcomes and waveforms are not read. Validation supports this exploratory comparison, not a test-set claim. We do not tune a threshold; the 0.5 threshold is fixed for threshold metrics.

In [1]:
from pathlib import Path
import csv, os, random, sys, platform, importlib.util, json, shutil
import numpy as np, pandas as pd, wfdb, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve)
from IPython.display import display

ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Data/AttributesDictionary.csv').is_file())
OUT=ROOT/'new work/Week5/experimental/seed_replication'; RECORDS=ROOT/'Child_ECG/Child_ecg'
W4=ROOT/'NoteBooks/Week4'; W5=ROOT/'NoteBooks/Week5'
SEED=42; FS=500; N_SAMPLES=5000; BATCH_SIZE=32; MAX_EPOCHS=8; PATIENCE=3; LR=3e-4; WEIGHT_DECAY=1e-4
assert Path(sys.executable).resolve()==(ROOT/'venv/bin/python').resolve(), 'Use existing project venv.'
print('Python',platform.python_version(),sys.executable)
print('Torch',torch.__version__,'compiled CUDA',torch.version.cuda,'CUDA available',torch.cuda.is_available())
assert torch.cuda.is_available(), 'Stop before training if CUDA is unavailable in the actual notebook kernel.'
DEVICE=torch.device('cuda');print('GPU',torch.cuda.get_device_name(0))
print('WFDB',wfdb.__version__,'pandas',pd.__version__,'NumPy',np.__version__)
torch.set_num_threads(min(4,os.cpu_count() or 1))
def seed_all():
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
seed_all()

Python 3.13.5 /home/souhit/Documents/Research/Pediatric_ECG/venv/bin/python
Torch 2.6.0+cu124 compiled CUDA 12.4 CUDA available True
GPU NVIDIA GeForce RTX 3060 Laptop GPU
WFDB 4.3.1 pandas 2.3.3 NumPy 2.5.2


In [2]:
metadata_path=ROOT/'Data/AttributesDictionary.csv'
with metadata_path.open(newline='') as handle:
    reader=csv.DictReader(handle)
    patient_ids=sorted({row['Patient_ID'] for row in reader})
random.Random(SEED).shuffle(patient_ids)
n_train=int(.70*len(patient_ids));n_val=int(.15*len(patient_ids))
train_ids=set(patient_ids[:n_train]);val_ids=set(patient_ids[n_train:n_train+n_val]);heldout_ids=set(patient_ids[n_train+n_val:])
assert not (train_ids&val_ids or train_ids&heldout_ids or val_ids&heldout_ids)
disease=pd.read_csv(ROOT/'Data/DiseaseCode.csv')
selected_codes=set(disease.loc[disease['Disease Type']!='Other diseases(OD)','ICD-10 Code'].astype(str).str.strip())
assert len(selected_codes)==19
def icd_tokens(value):return [x.strip().strip("'\"").strip() for x in value.split(';') if x.strip()]
train_rows=[];val_rows=[]
with metadata_path.open(newline='') as handle:
    for row in csv.DictReader(handle):
        pid=row['Patient_ID']
        if pid in heldout_ids:continue  # No test ICD field is accessed.
        assert row['Filename'].split('/')[-2]==pid
        row['target']=int(bool(set(icd_tokens(row['ICD-10 code']))&selected_codes))
        (train_rows if pid in train_ids else val_rows).append(row)
train_df=pd.DataFrame(train_rows);val_df=pd.DataFrame(val_rows)
for frame in [train_df,val_df]:
    frame['target']=frame.target.astype(int)
    frame['Lead']=frame.Lead.astype(int)
    frame['Sampling_point']=frame.Sampling_point.astype(int)
    frame['age_years']=frame.Age.str.extract(r'^(\d+)d$')[0].astype(float)/365.25
    assert frame.age_years.notna().all()
assert (len(train_df),len(val_df))==(9943,2169)
assert (train_df.target.sum(),val_df.target.sum())==(2595,583)
print('Patient counts: train',len(train_ids),'validation',len(val_ids),'reserved',len(heldout_ids))
print('Record counts: train',len(train_df),'validation',len(val_df))
print('Positive prevalence: train',round(train_df.target.mean(),4),'validation',round(val_df.target.mean(),4))
profile=pd.concat([train_df,val_df],ignore_index=True)
print('Lead counts train+validation',profile.Lead.value_counts().sort_index().to_dict())
print('Age years quantiles',profile.age_years.quantile([0,.25,.5,.75,1]).round(2).to_dict())
print('Duration seconds quantiles',(profile.Sampling_point/FS).quantile([0,.25,.5,.75,1]).to_dict())
print('Records longer than 10 s',int((profile.Sampling_point>N_SAMPLES).sum()))
print('Train+validation selected-code frequencies can overlap:')
code_frequency={code:sum(code in icd_tokens(value) for value in profile['ICD-10 code']) for code in sorted(selected_codes)}
display(pd.Series(code_frequency,name='records').sort_values(ascending=False).to_frame())
profile_summary=pd.DataFrame([{'split':'train','patients':len(train_ids),'records':len(train_df),'positive_records':int(train_df.target.sum()),'positive_prevalence':train_df.target.mean()},
                              {'split':'validation','patients':len(val_ids),'records':len(val_df),'positive_records':int(val_df.target.sum()),'positive_prevalence':val_df.target.mean()}])
profile_summary.to_csv(OUT/'cohort_summary.csv',index=False)

Patient counts: train 8150 validation 1746 reserved 1747
Record counts: train 9943 validation 2169
Positive prevalence: train 0.261 validation 0.2688
Lead counts train+validation {9: 1579, 12: 10533}
Age years quantiles {0.0: 0.0, 0.25: 4.25, 0.5: 8.59, 0.75: 12.38, 1.0: 14.99}
Duration seconds quantiles {0.0: 5.0, 0.25: 21.0, 0.5: 30.0, 0.75: 30.0, 1.0: 120.0}
Records longer than 10 s 12092
Train+validation selected-code frequencies can overlap:


,records
Q21.0,919
Q21.1,798
(FO) Q21.1,524
Q25.0,294
(V) I40.0,273
M30.3,159
I51.4,141
Q21.3,82
(F) I40.0,80
(OSD) Q21.1,75


In [3]:
LEADS=['I','II','III','AVR','AVL','AVF','V1','V2','V3','V4','V5','V6']
LEAD_INDEX={lead:i for i,lead in enumerate(LEADS)}
length_by_file=dict(zip(profile.Filename,profile.Sampling_point))
class CVDRecordDataset(Dataset):
    def __init__(self,frame):self.frame=frame[['Filename','target']].reset_index(drop=True)
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        signal,fields=wfdb.rdsamp(str(RECORDS/row.Filename),sampto=min(N_SAMPLES,int(length_by_file[row.Filename])))
        assert fields['fs']==FS and np.isfinite(signal).all()
        output=np.zeros((12,N_SAMPLES),dtype=np.float32)
        for j,lead in enumerate(fields['sig_name']):
            key=lead.upper();assert key in LEAD_INDEX
            x=signal[:,j].astype(np.float32);std=x.std()
            output[LEAD_INDEX[key],:len(x)]=(x-x.mean())/std if std>1e-8 else 0
        return torch.from_numpy(output),int(row.target)
train_data=CVDRecordDataset(train_df);val_data=CVDRecordDataset(val_df)
val_loader=DataLoader(val_data,batch_size=BATCH_SIZE,shuffle=False,num_workers=2)
def import_module(name,path):
    spec=importlib.util.spec_from_file_location(name,path)
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    return module
SimpleECGModel=import_module('week4_original_arch',W4/'pytorch_starter.py').SimpleECGModel
ResNet1D=import_module('week5_original_arch',W5/'models.py').ResNet1D
MultiScaleResNet1D=import_module('week8_multiscale_arch',ROOT/'NoteBooks/Week8/ExperimentB/models.py').MultiScaleResNet1D

def metrics(y,p):
    pred=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(accuracy=accuracy_score(y,pred),balanced_accuracy=balanced_accuracy_score(y,pred),
                precision=precision_score(y,pred,zero_division=0),sensitivity=recall_score(y,pred,zero_division=0),
                specificity=tn/(tn+fp),f1=f1_score(y,pred,zero_division=0),
                auroc=roc_auc_score(y,p),average_precision=average_precision_score(y,p),
                tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))

def evaluate_model(model,loader):
    model.eval();ys=[];ps=[];total=0.
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE);y=y.to(DEVICE);logits=model(x)
            total+=F.cross_entropy(logits,y).item()*len(y)
            ys.extend(y.cpu().tolist());ps.extend(torch.softmax(logits,dim=1)[:,1].cpu().tolist())
    return total/len(loader.dataset),np.asarray(ys),np.asarray(ps)



In [4]:
import time, gc
SEEDS=[42,43,44]
ARCHS={'resnet1d':ResNet1D,'multiscale_resnet':MultiScaleResNet1D}
config={'seeds':SEEDS,'seed_for_patient_split':SEED,'epochs_max':MAX_EPOCHS,'patience':PATIENCE,
        'batch_size':BATCH_SIZE,'learning_rate':LR,'weight_decay':WEIGHT_DECAY,
        'loss':'unweighted cross entropy','checkpoint_rule':'minimum validation cross entropy, improvement > 1e-5',
        'threshold':0.5,'input_samples':N_SAMPLES,'fs':FS,'lead_order':LEADS,
        'selected_codes':sorted(selected_codes),
        'architecture_sources':{'resnet1d':'NoteBooks/Week5/models.py:ResNet1D',
                                'multiscale_resnet':'NoteBooks/Week8/ExperimentB/models.py:MultiScaleResNet1D'},
        'test_evaluated':False}
(OUT/'config.json').write_text(json.dumps(config,indent=2))
def seed_run(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False

def run_one(name,cls,seed):
    seed_run(seed)
    model=cls(num_classes=2).to(DEVICE)
    # Independent generator fixes the sample order for both architectures at this seed.
    generator=torch.Generator().manual_seed(seed)
    train_loader=DataLoader(train_data,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,generator=generator)
    optimizer=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    history=[];best_loss=float('inf');best_epoch=None;stale=0
    stem=f'{name}_seed{seed}'
    history_path=OUT/f'{stem}_history.csv';checkpoint=OUT/f'{stem}_best.pth'
    for epoch in range(1,MAX_EPOCHS+1):
        start=time.monotonic();model.train();train_total=0.
        for x,y in train_loader:
            x=x.to(DEVICE);y=y.to(DEVICE)
            optimizer.zero_grad();logits=model(x);loss=F.cross_entropy(logits,y)
            loss.backward();optimizer.step();train_total+=loss.item()*len(y)
        train_loss=train_total/len(train_data)
        val_loss,y,p=evaluate_model(model,val_loader);m=metrics(y,p)
        row={'epoch':epoch,'train_loss':train_loss,'val_loss':val_loss,
             'val_accuracy':m['accuracy'],'val_sensitivity':m['sensitivity'],
             'val_specificity':m['specificity'],'val_f1':m['f1'],
             'val_auroc':m['auroc'],'val_average_precision':m['average_precision'],
             'elapsed_seconds':time.monotonic()-start}
        history.append(row);pd.DataFrame(history).to_csv(history_path,index=False)
        if val_loss<best_loss-1e-5:
            best_loss=val_loss;best_epoch=epoch;stale=0
            torch.save({'model_state_dict':model.state_dict(),'architecture':name,'seed':seed,
                        'epoch':epoch,'validation_loss':val_loss,'selected_codes':sorted(selected_codes),
                        'lead_order':LEADS,'input_samples':N_SAMPLES,'batch_size':BATCH_SIZE,
                        'lr':LR,'weight_decay':WEIGHT_DECAY,
                        'selection':'minimum unweighted validation cross entropy'},checkpoint)
        else:stale+=1
        print(f'{stem} epoch {epoch}: train {train_loss:.4f} val {val_loss:.4f} '
              f'AP {m["average_precision"]:.4f} AUROC {m["auroc"]:.4f} '
              f'sens {m["sensitivity"]:.4f} spec {m["specificity"]:.4f} '
              f'time {row["elapsed_seconds"]:.1f}s',flush=True)
        if stale>=PATIENCE:break
    saved=torch.load(checkpoint,map_location=DEVICE,weights_only=False)
    model.load_state_dict(saved['model_state_dict'])
    val_loss,y,p=evaluate_model(model,val_loader);m=metrics(y,p)
    assert np.array_equal(y,val_df.target.to_numpy())
    result={'architecture':name,'seed':seed,'epoch':best_epoch,'val_loss':val_loss,**m}
    del model,optimizer,train_loader;gc.collect();torch.cuda.empty_cache()
    return result,p


In [5]:
# Seed 42: train the standard ResNet, then the multi-scale ResNet, sequentially.
for name,cls in ARCHS.items():
    result,p=run_one(name,cls,42)
    print('selected',result,flush=True)
    summary_path=OUT/'validation_summary.csv'
    prior_rows=pd.read_csv(summary_path).to_dict('records') if summary_path.exists() else []
    prior_rows=[r for r in prior_rows if not (r['architecture']==name and int(r['seed'])==42)]
    pd.DataFrame(prior_rows+[result]).to_csv(summary_path,index=False)
    pd.DataFrame({'validation_index':np.arange(len(val_df)),
                  'patient_id':val_df.Patient_ID.to_numpy(),
                  'target':val_df.target.to_numpy(),
                  'age_years':val_df.age_years.to_numpy(),
                  'lead_count':val_df.Lead.to_numpy(),
                  'probability':p}).to_csv(OUT/f'{name}_seed42_predictions.csv',index=False)
print('Seed 42 pair complete.',flush=True)

resnet1d_seed42 epoch 1: train 0.5111 val 0.4828 AP 0.6211 AUROC 0.7760 sens 0.2727 spec 0.9735 time 18.2s


resnet1d_seed42 epoch 2: train 0.4826 val 0.4533 AP 0.6682 AUROC 0.7929 sens 0.3877 spec 0.9584 time 19.5s


resnet1d_seed42 epoch 3: train 0.4707 val 0.4454 AP 0.6708 AUROC 0.8003 sens 0.4357 spec 0.9458 time 20.3s


resnet1d_seed42 epoch 4: train 0.4629 val 0.4394 AP 0.6768 AUROC 0.8143 sens 0.4460 spec 0.9376 time 21.4s


resnet1d_seed42 epoch 5: train 0.4538 val 0.4416 AP 0.6976 AUROC 0.8153 sens 0.5540 spec 0.9105 time 21.7s


resnet1d_seed42 epoch 6: train 0.4468 val 0.4276 AP 0.6963 AUROC 0.8212 sens 0.4477 spec 0.9515 time 22.0s


... (8 outputs left out)

multiscale_resnet_seed42 epoch 6: train 0.4374 val 0.4313 AP 0.6915 AUROC 0.8199 sens 0.5009 spec 0.9369 time 26.7s


multiscale_resnet_seed42 epoch 7: train 0.4272 val 0.4514 AP 0.6691 AUROC 0.8034 sens 0.4563 spec 0.9426 time 25.6s


selected {'architecture': 'multiscale_resnet', 'seed': 42, 'epoch': 4, 'val_loss': 0.43054453315494134, 'accuracy': 0.8197325956662056, 'balanced_accuracy': 0.7107673489516979, 'precision': 0.7651933701657458, 'sensitivity': 0.4751286449399657, 'specificity': np.float64(0.9464060529634301), 'f1': 0.5862433862433862, 'auroc': 0.8170743577486541, 'average_precision': 0.6920290622929628, 'tn': 1501, 'fp': 85, 'fn': 306, 'tp': 277}


Seed 42 pair complete.


In [6]:
# Seed 43: train the standard ResNet, then the multi-scale ResNet, sequentially.
for name,cls in ARCHS.items():
    result,p=run_one(name,cls,43)
    print('selected',result,flush=True)
    summary_path=OUT/'validation_summary.csv'
    prior_rows=pd.read_csv(summary_path).to_dict('records') if summary_path.exists() else []
    prior_rows=[r for r in prior_rows if not (r['architecture']==name and int(r['seed'])==43)]
    pd.DataFrame(prior_rows+[result]).to_csv(summary_path,index=False)
    pd.DataFrame({'validation_index':np.arange(len(val_df)),
                  'patient_id':val_df.Patient_ID.to_numpy(),
                  'target':val_df.target.to_numpy(),
                  'age_years':val_df.age_years.to_numpy(),
                  'lead_count':val_df.Lead.to_numpy(),
                  'probability':p}).to_csv(OUT/f'{name}_seed43_predictions.csv',index=False)
print('Seed 43 pair complete.',flush=True)

resnet1d_seed43 epoch 1: train 0.5062 val 0.4891 AP 0.6553 AUROC 0.7944 sens 0.5952 spec 0.8506 time 25.3s


resnet1d_seed43 epoch 2: train 0.4795 val 0.4630 AP 0.6653 AUROC 0.8023 sens 0.2899 spec 0.9805 time 25.2s


resnet1d_seed43 epoch 3: train 0.4703 val 0.4671 AP 0.6831 AUROC 0.8063 sens 0.3328 spec 0.9754 time 23.8s


resnet1d_seed43 epoch 4: train 0.4604 val 0.4436 AP 0.6912 AUROC 0.8158 sens 0.4099 spec 0.9666 time 23.6s


resnet1d_seed43 epoch 5: train 0.4533 val 0.4966 AP 0.6570 AUROC 0.7979 sens 0.2367 spec 0.9842 time 24.3s


resnet1d_seed43 epoch 6: train 0.4473 val 0.4363 AP 0.6892 AUROC 0.8114 sens 0.4820 spec 0.9332 time 23.4s


... (8 outputs left out)

multiscale_resnet_seed43 epoch 6: train 0.4385 val 0.4775 AP 0.6818 AUROC 0.8079 sens 0.6123 spec 0.8480 time 26.6s


multiscale_resnet_seed43 epoch 7: train 0.4287 val 0.4504 AP 0.6957 AUROC 0.8194 sens 0.3242 spec 0.9823 time 26.7s


selected {'architecture': 'multiscale_resnet', 'seed': 43, 'epoch': 4, 'val_loss': 0.4372032415542585, 'accuracy': 0.8155832180728446, 'balanced_accuracy': 0.7106418944495034, 'precision': 0.7401574803149606, 'sensitivity': 0.483704974271012, 'specificity': np.float64(0.937578814627995), 'f1': 0.5850622406639004, 'auroc': 0.8173760974565181, 'average_precision': 0.6876578093151391, 'tn': 1487, 'fp': 99, 'fn': 301, 'tp': 282}


Seed 43 pair complete.


In [7]:
# Seed 44: train the standard ResNet, then the multi-scale ResNet, sequentially.
for name,cls in ARCHS.items():
    result,p=run_one(name,cls,44)
    print('selected',result,flush=True)
    summary_path=OUT/'validation_summary.csv'
    prior_rows=pd.read_csv(summary_path).to_dict('records') if summary_path.exists() else []
    prior_rows=[r for r in prior_rows if not (r['architecture']==name and int(r['seed'])==44)]
    pd.DataFrame(prior_rows+[result]).to_csv(summary_path,index=False)
    pd.DataFrame({'validation_index':np.arange(len(val_df)),
                  'patient_id':val_df.Patient_ID.to_numpy(),
                  'target':val_df.target.to_numpy(),
                  'age_years':val_df.age_years.to_numpy(),
                  'lead_count':val_df.Lead.to_numpy(),
                  'probability':p}).to_csv(OUT/f'{name}_seed44_predictions.csv',index=False)
print('Seed 44 pair complete.',flush=True)

resnet1d_seed44 epoch 1: train 0.5082 val 0.4654 AP 0.6610 AUROC 0.7900 sens 0.4854 spec 0.9187 time 26.7s


resnet1d_seed44 epoch 2: train 0.4832 val 0.4674 AP 0.6617 AUROC 0.7918 sens 0.3173 spec 0.9748 time 26.5s


resnet1d_seed44 epoch 3: train 0.4755 val 0.4442 AP 0.6902 AUROC 0.8168 sens 0.3345 spec 0.9729 time 24.2s


resnet1d_seed44 epoch 4: train 0.4640 val 0.4469 AP 0.6867 AUROC 0.8008 sens 0.3928 spec 0.9691 time 24.1s


resnet1d_seed44 epoch 5: train 0.4576 val 0.4553 AP 0.6836 AUROC 0.8101 sens 0.3808 spec 0.9653 time 24.9s


resnet1d_seed44 epoch 6: train 0.4526 val 0.4396 AP 0.6869 AUROC 0.8100 sens 0.4357 spec 0.9445 time 23.6s


... (9 outputs left out)

multiscale_resnet_seed44 epoch 7: train 0.4217 val 0.4269 AP 0.7026 AUROC 0.8244 sens 0.4751 spec 0.9369 time 27.2s


multiscale_resnet_seed44 epoch 8: train 0.4120 val 0.4441 AP 0.6749 AUROC 0.8152 sens 0.5180 spec 0.9161 time 27.6s


selected {'architecture': 'multiscale_resnet', 'seed': 44, 'epoch': 7, 'val_loss': 0.4268803727783438, 'accuracy': 0.8128169663439373, 'balanced_accuracy': 0.7060384712720005, 'precision': 0.7347480106100795, 'sensitivity': 0.4751286449399657, 'specificity': np.float64(0.9369482976040353), 'f1': 0.5770833333333333, 'auroc': 0.8243810009971468, 'average_precision': 0.7025699604838069, 'tn': 1486, 'fp': 100, 'fn': 306, 'tp': 277}


Seed 44 pair complete.


In [8]:
summary=pd.read_csv(OUT/'validation_summary.csv')
assert len(summary)==2*len(SEEDS)
assert not summary[['architecture','seed']].duplicated().any()
summary=summary.sort_values(['seed','architecture']).reset_index(drop=True)
summary.to_csv(OUT/'validation_summary.csv',index=False)
display(summary.round(4))
metric_names=['accuracy','sensitivity','specificity','f1','auroc','average_precision']
paired=[]
for seed in SEEDS:
    a=summary[(summary.seed==seed)&(summary.architecture=='resnet1d')].iloc[0]
    b=summary[(summary.seed==seed)&(summary.architecture=='multiscale_resnet')].iloc[0]
    paired.append({'seed':seed,**{k+'_multiscale_minus_resnet':b[k]-a[k] for k in metric_names}})
paired=pd.DataFrame(paired);paired.to_csv(OUT/'paired_seed_differences.csv',index=False)
display(paired.round(4))
# Descriptive age/lead subgroups, with n and positive prevalence for every run.
strata={'9 leads':val_df.Lead.to_numpy()==9,'12 leads':val_df.Lead.to_numpy()==12,
        'age <2y':val_df.age_years.to_numpy()<2,
        'age 2–<6y':(val_df.age_years.to_numpy()>=2)&(val_df.age_years.to_numpy()<6),
        'age 6–<12y':(val_df.age_years.to_numpy()>=6)&(val_df.age_years.to_numpy()<12),
        'age >=12y':val_df.age_years.to_numpy()>=12}
sub=[]
for name in ARCHS:
    for seed in SEEDS:
        frame=pd.read_csv(OUT/f'{name}_seed{seed}_predictions.csv')
        assert np.array_equal(frame.target.to_numpy(),val_df.target.to_numpy())
        for label,mask in strata.items():
            y=frame.target.to_numpy()[mask];p=frame.probability.to_numpy()[mask]
            sub.append({'architecture':name,'seed':seed,'subgroup':label,'n':len(y),
                        'positive_n':int(y.sum()),'prevalence':y.mean(),**metrics(y,p)})
sub=pd.DataFrame(sub);sub.to_csv(OUT/'validation_subgroups.csv',index=False)
display(sub[['architecture','seed','subgroup','n','positive_n','prevalence','sensitivity','specificity','auroc','average_precision']].round(4))

,architecture,seed,epoch,val_loss,accuracy,balanced_accuracy,precision,sensitivity,specificity,f1,auroc,average_precision,tn,fp,fn,tp
0,multiscale_resnet,42,4,0.4305,0.8197,0.7108,0.7652,0.4751,0.9464,0.5862,0.8171,0.6920,1501,85,306,277
1,resnet1d,42,6,0.4276,0.8160,0.6996,0.7722,0.4477,0.9515,0.5668,0.8212,0.6963,1509,77,322,261
2,multiscale_resnet,43,4,0.4372,0.8156,0.7106,0.7402,0.4837,0.9376,0.5851,0.8174,0.6877,1487,99,301,282
3,resnet1d,43,8,0.4352,0.8114,0.6904,0.7669,0.4288,0.9521,0.5501,0.8137,0.6902,1510,76,333,250
4,multiscale_resnet,44,7,0.4269,0.8128,0.7060,0.7347,0.4751,0.9369,0.5771,0.8244,0.7026,1486,100,306,277
5,resnet1d,44,7,0.4358,0.8137,0.7175,0.7157,0.5094,0.9256,0.5952,0.8181,0.6990,1468,118,286,297


,seed,accuracy_multiscale_minus_resnet,sensitivity_multiscale_minus_resnet,specificity_multiscale_minus_resnet,f1_multiscale_minus_resnet,auroc_multiscale_minus_resnet,average_precision_multiscale_minus_resnet
0,42,0.0037,0.0274,-0.0050,0.0195,-0.0041,-0.0043
1,43,0.0041,0.0549,-0.0145,0.0350,0.0037,-0.0026
2,44,-0.0009,-0.0343,0.0113,-0.0181,0.0062,0.0035


,architecture,seed,subgroup,n,positive_n,prevalence,sensitivity,specificity,auroc,average_precision
0,resnet1d,42,9 leads,281,157,0.5587,0.6879,0.7339,0.7944,0.8334
1,resnet1d,42,12 leads,1888,426,0.2256,0.3592,0.9699,0.7985,0.6283
2,resnet1d,42,age <2y,272,145,0.5331,0.6414,0.7717,0.8007,0.8269
3,resnet1d,42,age 2–<6y,495,212,0.4283,0.5189,0.9470,0.8229,0.8167
4,resnet1d,42,age 6–<12y,802,136,0.1696,0.3529,0.9745,0.7947,0.5416
5,resnet1d,42,age >=12y,600,90,0.1500,0.1111,0.9686,0.7059,0.3288
6,resnet1d,43,9 leads,281,157,0.5587,0.6433,0.8065,0.8121,0.8608
7,resnet1d,43,12 leads,1888,426,0.2256,0.3498,0.9644,0.7883,0.6095
8,resnet1d,43,age <2y,272,145,0.5331,0.6138,0.8425,0.8116,0.8513
9,resnet1d,43,age 2–<6y,495,212,0.4283,0.4670,0.9435,0.8207,0.8018


## Interpretation and limits

**Measured result on the same 2,169 validation records** (583 selected-code positives; 26.88% prevalence): all six checkpoints were selected by minimum validation cross entropy. Across seeds 42/43/44, the multi-scale minus standard ResNet AP differences were **−0.0043, −0.0026, +0.0035**; AUROC differences were **−0.0041, +0.0037, +0.0062**. Thus the earlier single-run multi-scale AP advantage of +0.0024 did **not** repeat consistently under this matched retraining protocol. Mean AP across the three runs was 0.6941 (multi-scale) versus 0.6952 (standard); these means describe only three seeds on one validation set and are not estimates of population performance.

At the fixed 0.5 threshold, multi-scale sensitivity differences were +0.0274, +0.0549, and −0.0343; F1 differences were +0.0195, +0.0350, and −0.0181. Specificity changed by −0.0050, −0.0145, and +0.0113. The tradeoff also changed direction at seed 44. The age ≥12 subgroup (n=600, 90 positives, prevalence 15%) had low sensitivity across all six retained checkpoints (0.111–0.167), while its AP varied 0.293–0.340. The full subgroup file reports n and prevalence for every age and lead-count group.

**Conclusion.** These runs do not support a reliable architectural improvement in AP, AUROC, or disease detection from multi-scale convolutions over the standard ResNet. The earlier multi-scale checkpoint remains a valid *selected experimental checkpoint* under its original validation criterion; this replication weakens any claim of general superiority. It does not justify replacing or discarding prior artifacts. Only validation patients were evaluated, the validation set had already been used in previous model selection, and three seeds on one split cannot establish generalization. No test-set record was loaded and no threshold was tuned.